In [119]:
import pandas as pd

df = pd.read_csv("../data/processed/repositories.csv", keep_default_na=False)

In [120]:
df.head(1)

,Repository Name,Description,Topics,Domain,Primary Language,Stars Count,Forks Count,Updated At,combined_text
0,tensorflow,An Open Source Machine Learning Framework for ...,"deep-learning, deep-neural-networks, distribut...",Machine Learning,C++,194622,75263,2026-04-10T10:08:00Z,an open source machine learning framework for ...


**Vectorizer**

In [121]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1,2),
    min_df= 2
)

X = vectorizer.fit_transform(df['combined_text'])

In [122]:
X.shape

(4997, 10000)

In [123]:
vectorizer.get_feature_names_out()[:50]

array(['10', '100', '100 clases', '100 days', '1000', '10000',
       '10000 gpus', '100daysofcode', '10x', '10x faster', '11', '12',
       '12 weeks', '13', '150', '16k', '180', '1wire', '1x', '1x 2x',
       '20', '2017', '2017 course', '2018', '2020', '2021', '2022',
       '2022 ai', '2023', '2024', '2024 agent', '2025', '2025 ai', '2026',
       '2026 codinginterviewquestions', '2026 论文和开源项目合集', '21', '229',
       '229 machine', '24', '24 lessons', '25', '26', '2d', '2d 3d',
       '2d game', '2dgame', '2dgameengine', '2fa', '2fa android'],
      dtype=object)

In [124]:
X[0].toarray()

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 10000))

In [125]:
from sklearn.metrics.pairwise import cosine_similarity

In [126]:
similarities = cosine_similarity(X[0], X)

In [127]:
similarities.shape

(1, 4997)

In [128]:
similarities[0][:10]

array([1.        , 0.1169294 , 0.09762156, 0.12870053, 0.07656773,
       0.21123698, 0.06808785, 0.07685919, 0.10708999, 0.10153275])

In [129]:
similarity_score = list(enumerate(similarities[0]))

In [130]:
similarity_score = sorted(
    similarity_score,
    key=lambda x: x[1],
    reverse=True
)

In [131]:
similarity_score[:10]

[(0, np.float64(1.0000000000000002)),
 (997, np.float64(0.9464730363635414)),
 (200, np.float64(0.9015566636896556)),
 (67, np.float64(0.4619608114749635)),
 (90, np.float64(0.43629175869886516)),
 (279, np.float64(0.3420496099164108)),
 (112, np.float64(0.3365582147991988)),
 (259, np.float64(0.3314341398637244)),
 (130, np.float64(0.32593509970094386)),
 (3832, np.float64(0.3223974268649733))]

In [132]:
indices = [i[0] for i in similarity_score[1:11]]

df.iloc[indices][["Repository Name", "Description", "Primary Language", "Stars Count"]]

,Repository Name,Description,Primary Language,Stars Count
997,tensorflow,An Open Source Machine Learning Framework for ...,C++,194622
200,tensorflow,An Open Source Machine Learning Framework for ...,C++,194622
67,handson-ml,⛔️ DEPRECATED – See https://github.com/ageron/...,Jupyter Notebook,25747
90,onnx,Open standard for machine learning interoperab...,Python,20630
279,onnx,Open standard for machine learning interoperab...,Python,20630
112,CNTK,"Microsoft Cognitive Toolkit (CNTK), an open so...",C++,17603
259,handson-ml,⛔️ DEPRECATED – See https://github.com/ageron/...,Jupyter Notebook,25747
130,kubeflow,Machine Learning Toolkit for Kubernetes,,15565
3832,onnx,Open standard for machine learning interoperab...,Python,20630
64,awesome-deep-learning-papers,The most cited deep learning papers,TeX,26106


**Recommendation Function**

In [133]:
# Recommendation Function

def recommend(repo_name, n=2):

    repo_indices = df.index[df["Repository Name"] == repo_name].tolist()

    if not repo_indices:
        return "Repository not found"

    idx = repo_indices[0]

    repo_vector = X[idx]

    similarities = cosine_similarity(repo_vector, X)[0] # 1-d array

    similarity_scores = list(enumerate(similarities))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the repo itself
    similarity_scores = similarity_scores[1:n+1]

    indices = [i[0] for i in similarity_scores]
    scores = [i[1] for i in similarity_scores]

    recommendations = df.iloc[indices][[
        "Repository Name",
        "Description",
        "Domain",
        "Primary Language",
        "Stars Count",
        "Forks Count",
        "Updated At"
    ]].copy()

    recommendations["Similarity"] = scores

    return recommendations


In [134]:
recommend('BitNet', 10)

,Repository Name,Description,Domain,Primary Language,Stars Count,Forks Count,Updated At,Similarity
416,awesome-llm-apps,Collection of awesome LLM apps with AI Agents ...,Python,Python,104972,15309,2026-04-10T10:09:54Z,0.233843
494,crewAI,"Framework for orchestrating role-playing, auto...",Python,Python,48506,6618,2026-04-10T10:13:51Z,0.227672
1156,transmission,Official Transmission BitTorrent client reposi...,C++,C++,14546,1367,2026-04-10T02:58:19Z,0.187202
1001,llama.cpp,LLM inference in C/C++,C++,C++,102889,16637,2026-04-10T10:18:15Z,0.182403
463,llama,Inference code for Llama models,Python,Python,59313,9833,2026-04-10T07:40:00Z,0.181132
436,vllm,A high-throughput and memory-efficient inferen...,Python,Python,75996,15412,2026-04-10T10:15:32Z,0.181025
417,DeepSeek-V3,,Python,Python,102543,16629,2026-04-10T09:43:01Z,0.179693
499,MediaCrawler,小红书笔记 | 评论爬虫、抖音视频 | 评论爬虫、快手视频 | 评论爬虫、B 站视频 ｜ 评...,Python,Python,47612,10236,2026-04-10T10:14:26Z,0.179693
567,jieba,结巴中文分词,Python,Python,34842,6705,2026-04-10T07:39:36Z,0.179693
570,TaskMatrix,,Python,Python,34173,3240,2026-04-10T07:44:12Z,0.179693


## EXPERIMENTS ##

**Checking Domain feature's influence**

In [138]:
df["text_without_domain"] = (
    df["Description"] + " " +
    df["Topics"]
)

df["text_with_domain"] = (
    df["Description"] + " " +
    df["Topics"] + " " +
    df["Domain"]
)

In [139]:
vectorizer_a = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

X_without_domain = vectorizer_a.fit_transform(
    df["text_without_domain"]
)

In [140]:
vectorizer_b = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

X_with_domain = vectorizer_b.fit_transform(
    df["text_with_domain"]
)

In [141]:
X_without_domain.shape


(4997, 10000)

In [142]:
X_with_domain.shape

(4997, 10000)

In [143]:
from sklearn.metrics.pairwise import cosine_similarity

bitnet_idx = df.index[
    df["Repository Name"] == "BitNet"
][0]

sim_without = cosine_similarity(
    X_without_domain[bitnet_idx],
    X_without_domain
)[0]

sim_with = cosine_similarity(
    X_with_domain[bitnet_idx],
    X_with_domain
)[0]

In [144]:
top_without = sim_without.argsort()[::-1][1:6]
top_with = sim_with.argsort()[::-1][1:6]

In [145]:
print("WITHOUT DOMAIN")
print(df.iloc[top_without][["Repository Name", "Domain"]])

print("\nWITH DOMAIN")
print(df.iloc[top_with][["Repository Name", "Domain"]])

WITHOUT DOMAIN
          Repository Name         Domain
2611  Reverse-Engineering  Cybersecurity
1156         transmission            C++
1001            llama.cpp            C++
463                 llama         Python
4695      agibot_x1_infer       Robotics

WITH DOMAIN
          Repository Name         Domain
416      awesome-llm-apps         Python
494                crewAI         Python
1156         transmission            C++
2611  Reverse-Engineering  Cybersecurity
463                 llama         Python


*Domain provides useful categorical information, but it shouldn't be the main source of similarity.*